In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning_scripts.lightning_ssl_matched_speech_in_noise import LitAudioSSL
from jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals

sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path
import pickle
from lightning_scripts.eval_jsin_transfer_matched import SSLClassifier

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/common.py:31: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


In [2]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/resnet18_barlow_invariant_only_lmbda_1e-2_lr_2e-1_w_invar_augment_no_avgpool.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

TASK = "word"
LAYER = "layer2" ## ckpt at this layerneeds to exist 

config['data'] = {}
config['data']['root'] = "/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/"
config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['data']['eval_max'] = 3
# config['hparas']['optimizer'] = args.optimizer
# config['hparas']['lr'] = args.lr * args.gpus
# config['hparas']['epochs'] = 2
# don't load in classifier head if it exists 
config['model']['arch_kwargs']['supervised'] =  False
config['model']['arch_kwargs']['time_average'] = False

if TASK == "word":
    config['data']['task_label'] = 'signal/word_int'
    config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794} 
    task_str = f"word_task"

elif TASK == "speaker":
    config['data']['task_label'] = 'signal/speaker_int'
    config['model']['arch_kwargs']['n_classes'] =  433

In [3]:
ckpt_path = "model_checkpoints/resnet18_barlow_invariant_only_lmbda_1e-2_lr_2e-1_w_invar_augment_no_avgpool/linear_classifier_checkpoints_word_task_layer2_full_rep_AdamW_1e-05_w_dropout/epoch=2-step=68400.ckpt"
classifier_ckpt = torch.load(ckpt_path, weights_only=False) # get latest checkpoint 
# dummy init with checkpoint 
module = SSLClassifier(config=config, ckpt_path=ckpt_path, layer_out=LAYER)


module.load_state_dict(classifier_ckpt['state_dict'])

module = module.cuda()
## update keys to remove _orig_mod from eatch key 

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:191: Found keys that are in the model state dict but not in the checkpoint: ['audio_rep.rep.downsampling_op.downsample_filter', 'audio_rep.rep.Cochleagram.compute_subbands.coch_filters', 'audio_rep.rep.Cochleagram.downsampling.downsample_filter', 'model.front_end.rep.downsampling_op.downsample_filter', 'model.front_end.rep.Cochleagram.compute_subbands.coch_filters', 'model.front_end.rep.Cochleagram.downsampling.downsample_filter', 'model.model.f.conv1.weight', 'model.model.f.bn1.weight', 'model.model.f.bn1.bias', 'model.model.f.bn1.running_mean', 'model.model.f.bn1.running_var', 'model.model.f.layer1.0.conv1.weight', 'model.model.f.layer1.0.bn1.weight', 'model.model.f.layer1.0.bn1.bias', 'model.model.f.layer1.0.bn1.running_mean', 'model.model.f.layer1.0.bn1.running_var', 'model.model.f.layer1.0.conv2.weight', 'model.model.f.layer1.0.bn2.weight', 'model.model.f.layer1.0.bn2.bias', 'mode

In [4]:
# run test 
test_dataset = jsinV3_precombined_all_signals(root=config['data']['root'],
                                                train=False,
                                                transform=None,
                                                batch_size=100,
                                                eval_max=1)
test_dataset.target_keys = ['signal/word_int']
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    num_workers=config['num_workers'],
    shuffle=False,
    collate_fn=module.eval_collate_fn
)


In [5]:
test_dataset

In [6]:
audio, labels = next(iter(test_dataloader))
word_labels = labels['signal/word_int']

In [7]:

batch = next(iter(test_dataloader))
with torch.no_grad():
    model_preds = module(audio.cuda())
    word_preds = model_preds['signal/word_int'].cpu().softmax(-1).cpu()
    model_top_1 = word_preds.argmax(-1)

In [8]:
raw_acc = (model_top_1 == word_labels).numpy().mean()
raw_acc

np.float64(0.47)

In [9]:
# model top5
top_5 = torch.isin(torch.topk(word_preds, k=5, dim=-1).indices, word_labels).any(-1).float().mean()
top_5

tensor(0.8300)

In [10]:
word_and_speaker_encodings = pickle.load(
    open("/mnt/home/igriffith/ceph/projects/cochdnn/robustness/audio_functions/word_and_speaker_encodings_jsinv3.pckl", "rb")
)
class_map = word_and_speaker_encodings["word_idx_to_word"]

In [11]:
### Geck examples where model predicted wrong label 




model_failure_IXS = torch.where((model_top_1 != word_labels))[0].numpy()

for _ in range(20):

    failure_eg = int(model_failure_IXS[_])

    true_word = class_map[int(word_labels[failure_eg])]
    ## Get model top 1 and top 5 for that eg 
    model_pred = class_map[int(model_top_1[failure_eg])]

    # model top 5 transcripbed 
    model_eg_top5 = torch.topk(word_preds[failure_eg], k=5, dim=-1).indices
    model_eg_top5_words = [class_map[int(ix)] for ix in model_eg_top5] 

    print(f"True word: {true_word}")
    print(f"Model top 5 words: {', '.join(model_eg_top5_words)}")
    display(Audio(audio[failure_eg], rate=20_000, normalize=False))
    print("\n")


True word: ended
Model top 5 words: considered, event, other, media, nuclear




True word: majority
Model top 5 words: another, majority, according, major, university




True word: london
Model top 5 words: among, under, front, problem, london




True word: separate
Model top 5 words: secretary, specific, significant, separate, support




True word: making
Model top 5 words: became, making, having, think, native


...



True word: college
Model top 5 words: military, current, century, federal, required




True word: during
Model top 5 words: nearly, turned, during, between, generally




True word: parts
Model top 5 words: possible, thousands, process, about, complex




True word: course
Model top 5 words: force, course, quarter, court, reports




True word: words
Model top 5 words: example, where, nineteen, according, current


tensor([158, 244, 485, 421, 463])